# AIC354 Assignment 2 — Task 2
## Heart Disease Prediction System

**Deployment Link:** `PASTE_YOUR_LIVE_CLOUD_URL_HERE`

The assignment requires the UCI Heart Disease dataset and explicitly constrains the prediction system to **only four input attributes**. This notebook uses exactly: `age`, `sex`, `cp`, and `trestbps`.

UCI reports 303 instances and 13 features for this dataset and documents the Cleveland processed data as the commonly used subset. The original target `num` ranges from 0 to 4; for binary prediction, values above 0 are mapped to 1.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE = Path.cwd()
if not (BASE / 'heart_cleveland.csv').exists() and (BASE / 'heart' / 'heart_cleveland.csv').exists():
    BASE = BASE / 'heart'
DATA_PATH = BASE / 'heart_cleveland.csv'
MODEL_PATH = BASE / 'heart_model.joblib'


## 1. Download / Load the UCI Dataset
If `heart_cleveland.csv` is not present, run `download_heart_data.py`. The code uses the UCI repository through `ucimlrepo`.


In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    from ucimlrepo import fetch_ucirepo
    heart_disease = fetch_ucirepo(id=45)
    X_raw = heart_disease.data.features.copy()
    y_raw = heart_disease.data.targets.copy()
    df = X_raw.copy()
    df['target'] = pd.to_numeric(y_raw.iloc[:,0], errors='coerce')
    df = df.dropna(subset=['target'])
    df['target'] = (df['target'] > 0).astype(int)
    df.to_csv(DATA_PATH, index=False)

print('Shape:', df.shape)
display(df.head())


In [ ]:
print('Missing values:', df.isnull().sum())
print('Target distribution:', df['target'].value_counts())


## 2. Select Exactly Four Input Attributes
The assignment constraint is satisfied here: the feature matrix contains exactly four columns.


In [ ]:
FEATURES = ['age', 'sex', 'cp', 'trestbps']
X = df[FEATURES].copy()
y = df['target'].astype(int)
print('Number of input attributes:', X.shape[1])
print('Input attributes:', FEATURES)


## 3. Preprocessing
The four variables are treated as numeric inputs for this compact assignment model. Missing numeric values are imputed with the median and then standardized.


In [ ]:
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


## 4. Train the Prediction Model


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])
model.fit(X_train, y_train)


## 5. Evaluate the Model


In [ ]:
y_pred = model.predict(X_test)
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall   :', round(recall_score(y_test, y_pred), 4))
print('F1-score :', round(f1_score(y_test, y_pred), 4))
print('Classification report:')
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Heart Disease Confusion Matrix')
plt.show()


## 6. Save the Trained Model


In [ ]:
joblib.dump(model, MODEL_PATH)
print('Model saved to:', MODEL_PATH)


## 7. Test a New Patient
Only the four required attributes are accepted.


In [ ]:
sample = pd.DataFrame([{
    'age': 55, 'sex': 1, 'cp': 2, 'trestbps': 130
}])
pred = int(model.predict(sample)[0])
prob = float(model.predict_proba(sample)[0,1])
print('Prediction:', 'Heart disease' if pred else 'No heart disease')
print('Estimated probability:', round(prob, 4))


## 8. Deployment
The included `app.py` uses the same four inputs and loads `heart_model.joblib`. Deploy the `heart` folder to a cloud platform such as Streamlit Cloud. After deployment, replace the deployment link at the top with the real URL.
